# 4.5 · Lasso 回归 / Lasso (L1)

> **课程定位 / Where this fits**
> **Part 4 第 5 课**。Ridge(4.4) 用 L2 收缩但不为零; Lasso 把惩罚换成 **L1**, 神奇地让部分系数**精确变 0** = **自动特征选择**。本课的灵魂问题: **为什么 L1 能稀疏而 L2 不能？**（几何 + 软阈值双解释）。2.10 节: Lasso = Laplace 先验 MAP。
> Lasso swaps L2 for L1 and zeroes out coefficients — automatic feature selection. The soul question: why does L1 sparsify but L2 doesn't?

> 💡 **面试相关 / Interview-relevant**
> - "L1 vs L2 正则化区别" ★★★★★（必考第一题）
> - "为什么 Lasso 能特征选择而 Ridge 不能" ★★★★★（几何解释）
> - "Lasso 为什么没有闭式解" ★★★（L1 不可导）
> - "Lasso 和 Ridge 怎么选" ★★★

---

## 学习目标 / Learning Objectives
1. 写出 Lasso 目标, 理解 L1 惩罚。
2. **几何解释**: 为什么 L1 的菱形约束产生稀疏解。
3. **软阈值**: 坐标下降的核心算子, 解释系数如何被"推到 0"。
4. 看 Lasso **系数路径**（系数逐个变 0）。
5. 用 Lasso 做特征选择 + 对照 Ridge。

## 目录 / TOC
1. [目标函数 + L1 vs L2 ⭐](#1)
2. [几何解释: 为什么 L1 稀疏 ⭐](#2)
3. [数据](#3)
4. [软阈值算子 + 坐标下降](#4)
5. [Lasso 系数路径: 系数逐个归零 ⭐](#5)
6. [特征选择实战](#6)
7. [= Laplace 先验 MAP (2.10)](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 目标函数 + L1 vs L2 ⭐ / Objective

$$\text{Lasso}: \quad J(\mathbf{w}) = \|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2 + \lambda \|\mathbf{w}\|_1, \qquad \|\mathbf{w}\|_1 = \sum_j |w_j|$$

和 Ridge 只差**范数**: $\|\mathbf{w}\|_2^2 = \sum w_j^2$ → $\|\mathbf{w}\|_1 = \sum|w_j|$。

| | Ridge (L2) | Lasso (L1) |
|---|---|---|
| 惩罚 | $\sum w_j^2$ | $\sum\|w_j\|$ |
| 效果 | 系数收缩但 ≠ 0 | 部分系数**精确 = 0** ⭐ |
| 特征选择 | ❌ | ✅ 自动 |
| 闭式解 | ✅ | ❌ (L1 在 0 处不可导) |
| 共线特征 | 平摊系数 | **任选一个**, 其余归 0 |
| 先验 (2.10) | 高斯 | Laplace (尖峰厚尾) |

**一句话**: **要稀疏/特征选择 → Lasso; 要稳定收缩/保留所有特征 → Ridge**。
Want sparsity/selection → Lasso; want stable shrinkage keeping all features → Ridge.


<a id="2"></a>
## 2. 几何解释: 为什么 L1 稀疏 ⭐ / Why L1 Sparsifies

**最经典的可视化**。把"最小化损失 + 惩罚"看成"在惩罚约束区域内最小化损失":
- 损失等高线是**椭圆**（OLS 解在中心）
- L2 约束区域是**圆**, L1 约束区域是**菱形(顶点在坐标轴上)**

最优解 = 椭圆**第一次碰到**约束区域的点：
- **L2 (圆)**: 碰点几乎总在圆的"光滑边"上 → 两个坐标都非零
- **L1 (菱形)**: 碰点**极易落在尖角(坐标轴)上** → 一个坐标 = 0 = 稀疏！

**菱形的尖角在坐标轴上**, 这就是 L1 产生稀疏的几何本质。
The diamond's corners lie on the axes, so the ellipse tends to touch there — one coordinate becomes exactly zero.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
w1 = np.linspace(-2, 3, 400); w2 = np.linspace(-2, 3, 400)
W1, W2 = np.meshgrid(w1, w2)
# 损失等高线 (椭圆, OLS 解在 (1.8, 1.2)) / loss ellipses
center = np.array([1.8, 1.2])
A = np.array([[2, 0.8], [0.8, 1]])
Loss = A[0,0]*(W1-center[0])**2 + 2*A[0,1]*(W1-center[0])*(W2-center[1]) + A[1,1]*(W2-center[1])**2

for ax, norm, t in [(axes[0], "L2 (Ridge): 圆", "圆"), (axes[1], "L1 (Lasso): 菱形", "菱形")]:
    ax.contour(W1, W2, Loss, levels=12, cmap="Blues", alpha=0.6)
    theta = np.linspace(0, 2*np.pi, 200)
    if "L2" in norm:
        ax.plot(1.2*np.cos(theta), 1.2*np.sin(theta), "r-", lw=2)   # 圆
        ax.scatter([1.05], [0.55], c="red", s=120, zorder=5, label="解(两坐标非零)")
    else:
        ax.plot([1.2,0,-1.2,0,1.2], [0,1.2,0,-1.2,0], "r-", lw=2)    # 菱形
        ax.scatter([0], [1.2], c="red", s=150, marker="*", zorder=5, label="解(w1=0, 稀疏!)")
    ax.scatter(*center, c="green", s=80, zorder=5, label="OLS 解")
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_xlabel("w1"); ax.set_ylabel("w2"); ax.set_title(norm); ax.legend(fontsize=8)
    ax.set_aspect("equal")
plt.tight_layout(); plt.show()
print("L2: 椭圆碰圆的光滑边 → w1,w2 都非零")
print("L1: 椭圆碰菱形尖角(在 w2 轴上) → w1=0, 稀疏! 这就是 Lasso 特征选择的几何原理")


<a id="3"></a>
## 3. 数据 / Data


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X, y = data.data.values, data.target.values
# 加 20 个纯噪声特征, 看 Lasso 能不能筛掉它们 / add 20 noise features
n = len(X)
noise = rng.normal(size=(n, 20))
X_aug = np.c_[X, noise]
feat_names = list(data.feature_names) + [f"noise_{i}" for i in range(20)]

X_tr, X_te, y_tr, y_te = train_test_split(X_aug, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
print(f"8 真实特征 + 20 噪声特征 = {X_aug.shape[1]} 维; 看 Lasso 能否筛掉噪声")


<a id="4"></a>
## 4. 软阈值算子 + 坐标下降 / Soft-thresholding & Coordinate Descent

Lasso 没有闭式解（L1 在 0 处不可导）。主流解法是**坐标下降**: 一次只优化一个系数, 其他固定。每步的更新就是**软阈值算子**:

$$w_j \leftarrow S_\lambda(\rho_j) = \text{sign}(\rho_j)\cdot\max(|\rho_j| - \lambda, 0)$$

其中 $\rho_j$ 是 OLS 在该坐标的更新。**软阈值的关键**: 如果 $|\rho_j| \le \lambda$, 直接**把 $w_j$ 设为 0**——这就是系数被"推到 0"的算子级原因。
The soft-threshold operator zeroes out any coordinate whose OLS update is within λ of zero — the operator-level reason coefficients vanish.


In [ ]:
def soft_threshold(rho, lam):
    return np.sign(rho) * max(abs(rho) - lam, 0.0)

# 演示软阈值: 小信号被清零, 大信号被收缩 / small signals zeroed, large shrunk
rhos = np.linspace(-3, 3, 200)
lam = 1.0
out = [soft_threshold(r, lam) for r in rhos]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(rhos, out, lw=2, label=f"软阈值 S_λ(ρ), λ={lam}")
ax.plot(rhos, rhos, "k--", alpha=0.4, label="无惩罚 (y=ρ)")
ax.axvspan(-lam, lam, alpha=0.15, color="red", label=f"|ρ|≤λ → 清零")
ax.axhline(0, color="gray", lw=0.5); ax.legend(); ax.set_xlabel("ρ"); ax.set_ylabel("w")
ax.set_title("软阈值: |ρ|≤λ 区间被压成 0 (死区) → 产生稀疏")
plt.tight_layout(); plt.show()
print("红区(|ρ|≤λ): 系数直接=0; 红区外: 系数向0收缩 λ. 这是 Lasso 稀疏的算子机制")


<a id="5"></a>
## 5. Lasso 系数路径: 系数逐个归零 ⭐ / Coefficient Path

对比 4.4 Ridge 的"平滑收缩不为零", Lasso 路径里**系数一个接一个精确变 0**——λ 越大, 存活特征越少。


In [ ]:
from sklearn.linear_model import Lasso

alphas = np.logspace(-3, 0.5, 60)
coef_path = np.array([Lasso(alpha=a, max_iter=5000).fit(Xtr, y_tr).coef_ for a in alphas])

fig, ax = plt.subplots(figsize=(9, 5))
for i in range(8):    # 真实特征 / real features
    ax.plot(alphas, coef_path[:, i], lw=1.8, label=feat_names[i])
for i in range(8, 28):  # 噪声特征 (灰色) / noise features
    ax.plot(alphas, coef_path[:, i], color="gray", alpha=0.3, lw=0.8)
ax.set_xscale("log"); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("λ (alpha)"); ax.set_ylabel("系数")
ax.legend(fontsize=7, ncol=2, title="彩色=真实特征, 灰=噪声")
ax.set_title("Lasso 路径: λ↑ 系数逐个精确归0 (噪声特征最先死)")
plt.tight_layout(); plt.show()
print("灰色噪声特征率先归0, 真实特征撑得更久 → Lasso 自动识别并剔除无用特征")


<a id="6"></a>
## 6. 特征选择实战 / Feature Selection in Action


In [ ]:
from sklearn.linear_model import LassoCV, Lasso, Ridge

# LassoCV 自动选 λ / LassoCV picks λ
lassocv = LassoCV(cv=5, max_iter=10000, random_state=0).fit(Xtr, y_tr)
n_zero = (np.abs(lassocv.coef_) < 1e-8).sum()
n_zero_noise = (np.abs(lassocv.coef_[8:]) < 1e-8).sum()
print(f"LassoCV 选出 λ = {lassocv.alpha_:.4f}")
print(f"系数为 0 的特征: {n_zero}/{X_aug.shape[1]} (其中 {n_zero_noise}/20 个噪声被剔除)")
print(f"存活的真实特征: {(np.abs(lassocv.coef_[:8]) > 1e-8).sum()}/8\n")

# Ridge 对照: 不归零 / Ridge keeps all
ridge = Ridge(alpha=1.0).fit(Xtr, y_tr)
print(f"对照 Ridge: 系数为 0 的特征 = {(np.abs(ridge.coef_) < 1e-8).sum()} (一个都不归零)")
print(f"\ntest R²: Lasso={lassocv.score(Xte,y_te):.4f}, Ridge={ridge.score(Xte,y_te):.4f}")
print("Lasso 自动剔除噪声 → 更简洁可解释的模型; 噪声多时 Lasso 常胜")


<a id="7"></a>
## 7. = Laplace 先验 MAP (2.10) / Lasso = Laplace-prior MAP

2.10 节证过: 给参数 **Laplace 先验** $p(w_j) \propto e^{-|w_j|/b}$, 做 MAP 估计 → **等价 Lasso**。

**为什么 Laplace 先验产生稀疏**: Laplace 分布在 0 处有**尖峰**（比高斯尖得多）+ 厚尾 → 先验强烈相信"系数恰好是 0", 偶尔允许大值。对比高斯先验（Ridge）在 0 处光滑 → 只收缩不归零。

**先验形状决定稀疏性**: 尖峰(Laplace)→稀疏, 光滑(高斯)→收缩。这把 4.4/4.5 统一在贝叶斯框架下。
The prior's shape determines sparsity: a spiked Laplace prior zeroes coefficients; a smooth Gaussian only shrinks.


<a id="8"></a>
## 8. 小结 / Summary

```
Lasso: min ‖y-Xw‖² + λ‖w‖₁  →  部分系数精确=0 = 自动特征选择 ⭐
为什么稀疏:
  几何: L1 菱形约束的尖角在坐标轴上 → 椭圆易碰尖角 → 坐标归0
  算子: 软阈值 S_λ(ρ) 把 |ρ|≤λ 直接清零 (坐标下降)
  贝叶斯: Laplace 先验在0处尖峰 (2.10)
无闭式解 (L1 不可导) → 坐标下降/LARS
λ↑ 系数逐个归零 (噪声先死); 共线特征任选其一
选择: 要稀疏/可解释→Lasso, 要稳定保留全特征→Ridge, 都想要→Elastic Net(4.6)
```

### 💡 面试速查
1. **L1 稀疏 L2 不稀疏**: 几何上 L1 菱形尖角在轴上
2. **Lasso = 自动特征选择**; Ridge 只收缩不归零
3. **软阈值**: |ρ|≤λ 清零 → 坐标下降的核心
4. **Lasso 无闭式解** (L1 在0不可导)
5. **Lasso=Laplace先验, Ridge=高斯先验** (2.10)

### 下一节
**4.6 弹性网 Elastic Net**——L1+L2 混合, 兼得 Lasso 的稀疏和 Ridge 的稳定, 尤其解决 Lasso 在共线特征上"任选一个"的随意性。
